In [1]:
%matplotlib widget
# Boilerplate import code for all libraries
# Changes to the precision require re-loading the kernel and need to be done before any op uses them.
import sphWarpCore_config as swc
from typing import Any
swc.configure(precision="float32", dim=Any) # precision: float16|half|float32|single|float64|double

import sphWarpCore as sph
from sphWarpCore.type_config import *
print(get_type_config()) # confirms active settings

# Initialize warp at this point
import warp as wp; wp.init()

import os
import torch
if torch.cuda.is_available(): # set the TORCH_CUDA_ARCH_LIST environment variable to the compute capability of the GPU for faster compiles
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}'

import warnings
from tqdm import TqdmExperimentalWarning
warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)
from tqdm.autonotebook import tqdm

# final import blocks that are generic
import matplotlib.pyplot as plt
from torch.profiler import profile, record_function, ProfilerActivity
import numpy as np
import math
import shlex    
import subprocess
import shutil

# custom SPH libraries
from integrators.integration import *
from sphWarpCore import *

# This library
from compressibleSPH import *

{'scalar_t': <class 'warp._src.types.float32'>, 'dim_t': typing.Any}
Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.2
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA RTX PRO 500 Blackwell Generation Laptop GPU" (6 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/lu26029/.cache/warp/1.12.0


In [2]:
import math
import numpy as np


nx = 128
dim = 2

L = 1
dx = L/nx
aspect = 2
band = 20

n_h = 4

gamma = 1.4
rho0 = 1.0
rho_low = 1
rho_high = 2



extraData = {
    'nx': nx,
    'dim': dim,
    'L': L,
    'n_h': n_h,

    'gamma': gamma,
    'rho0': rho0,
    'rho_low': rho_low,
    'rho_high': rho_high,

    'dx': dx,
    'aspect': aspect,
    'band': band

}

In [ ]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
dtype = get_torch_precision()


domain = buildDomainDescription(l = 1, dim = dim, periodic = True, device = device, dtype = dtype)
domain.min[0] = -L/2/aspect
domain.max[0] = L/2/aspect
domain.min[1] = 0 - band * dx
domain.max[1] = L + band * dx

config, integrator = buildConfig(
    domain = domain,
    dim = dim,
    kernel = KernelFunctions.B7,
    targetNeighbors = n_h_to_nH(4, dim),
    supportMode = SupportScheme.KernelMeanSymmetric,
    gradientMode = GradientScheme.Difference,
    laplacianMode = LaplacianScheme.Brookshaw,
    integrationScheme = IntegrationSchemeType.rungeKutta2,
    samplingScheme = SamplingScheme.regular,
    device = device,
    dtype = dtype,
    dt = None,
    adaptiveDt = True,
    cflFactor=0.3,
)
config.nx = nx

config.minDt = 1e-8
# config.dx = L / (nx * 2)

scheme = CompressibleSPHScheme.CRKSPH
SimulationSystem, SimulationState, SimulationConfig, SimulationUpdate, fn, export_fn, import_fn = buildScheme(scheme)


schemeConfig = SimulationConfig()
schemeConfig.gamma = gamma
schemeConfig.rho0 = rho0


schemeConfig.viscositySwitchParams.scheme = ViscositySwitch.NoneSwitch
schemeConfig.adaptiveSupportScheme = AdaptiveSupportScheme.Owen
schemeConfig.adaptiveSupportCorrections = False

In [4]:
config.domain.min[0] = -L/2/aspect
config.domain.max[0] = L/2/aspect
config.domain.min[1] = 0 - band * dx
config.domain.max[1] = L + band * dx

compressibleSystem = setupBasicCompressibleInitialState(nx, config, schemeConfig, SimulationState, SimulationSystem)



In [5]:
rho_b = rho_low
rho_t = rho_high
delta = 0.0025
g = 1/2
gamma = schemeConfig.gamma
P_0 = rho_t / gamma
print(gamma)


1.4


In [6]:
def rayleighTaylor_rho(positions, rho_b, rho_t, delta):
    # print('Enforcing Rayleigh-Taylor density for ', positions.shape[0], ' particles')
    y = positions[:,1]
    rho_y = rho_b + (rho_t - rho_b) * (1 + torch.exp(-(y - 0.5) / delta))**(-1)
    return rho_y
def rayleighTaylor_u(positions, rho_b, rho_t, delta, g, gamma):
    y = positions[:,1]
    rho_y = rayleighTaylor_rho(positions, rho_b, rho_t, delta)
    P_0 = rho_t / gamma
    P = P_0 - g * rho_y * (y - 1/2)
    u = P / rho_y / (gamma - 1)
    return u


positions = compressibleSystem.state.positions
x = positions[:,0]
y = positions[:,1]

# rho_y = rho_b + (rho_t - rho_b) * (1 + torch.exp(-(y - 0.5) / delta))**(-1)
rho_y = rayleighTaylor_rho(positions, rho_b, rho_t, delta)

delta_y = delta * 5
v_y = delta_y * (1 + torch.cos(8 * np.pi * (x + 0.25))) * (1 + torch.cos(5 * np.pi * (y - 0.5)))
v_y[~((y >= 0.3) & (y <= 0.7))] = 0

P = P_0 - g * rho_y * (y - 1/2)

Pinitial = P
rhoInitial = rho_y
vInitial = torch.stack([torch.zeros_like(v_y), v_y], dim=-1)

In [7]:



# particles_ = sampleRegularParticles(nx, config.domain, config.targetNeighbors)
# print(f"Sampled {particles_.positions.shape[0]} particles.")
# # domain = buildDomainDescription(domainExtent * 1.5, dim, periodic = periodicDomain, device = device, dtype = dtype)
# particles_ = particles_._replace(masses = particles_.masses * compressibleSPHConfig.rho0)


# print(f"Sampled {particles_.positions.shape[0]} particles.")
# print(f' Support: Min: {particles_.supports.min().item()}, Max: {particles_.supports.max().item()}, Mean: {particles_.supports.mean().item()}')

# particles = SimulationState(
#     positions = particles_.positions,
#     supports = particles_.supports,
#     masses = particles_.masses,
#     densities = particles_.densities,
#     velocities = torch.zeros_like(particles_.positions),
    
#     kinds = torch.zeros_like(particles_.positions[:,0], dtype = torch.int32),
#     materials = torch.zeros_like(particles_.positions[:,0], dtype = torch.int32),
#     UIDs = torch.arange(particles_.positions.shape[0], device = device, dtype = torch.int32),
#     UIDcounter = particles_.positions.shape[0],
    
#     internalEnergies = None,
#     totalEnergies = None,
#     entropies = None,
#     pressures = None,
#     soundspeeds = None,

#     divergence=torch.zeros_like(particles_.densities),
#     alpha0s= torch.ones_like(particles_.densities),
#     alphas= torch.ones_like(particles_.densities),
# )


# densities = warpOperation(
#     particles, 
#     OperationProperties(
#         kernel = config.kernel,
#         operation = WarpOperation.Density,
#         supportMode = SupportScheme.Gather,
#         gradientMode = config.gradientMode,
#         laplacianMode = config.laplacianMode,
#     ),
#     domain = config.domain,
# )
# particles.densities = densities

# compressibleSPHConfigAdaptiveH = CompressibleSPHConfig(
#     adaptiveSupportIterations=16,
#     adaptiveSupportThreshold=1e-3,
#     adaptiveSupportScheme=AdaptiveSupportScheme.Owen,
# )

# from compressibleSPH.modules import *

# # wrappedKernel = warpKernelToDiffSPHKernel(kernel)
# # neighborhood, neighbors = evaluateNeighborhood(particles, domain, wrappedKernel, verletScale = 1.0, mode = SupportScheme.SuperSymmetric, priorNeighborhood=None)
# # numNeighbors = coo_to_csr(filterNeighborhoodByKind(particles, neighbors.neighbors, which = 'noghost')).rowEntries
# # config = {'targetNeighbors': targetNeighbors, 'domain': domain, 'support': {'iterations': 16, 'scheme': 'Monaghan'}, 'neighborhood': {'algorithm': 'compact'}}
# # rho, h, rhos, hs, neighborhood = evaluateOptimalSupport(particles, wrappedKernel, neighborhood, SupportScheme.Gather, config)

# rho_optimal, h_optimal, adjacency, rhos_iter, supports_iter = evaluateOptimalSupport(particles, config, supportScheme = SupportScheme.Gather, compParams = compressibleSPHConfigAdaptiveH)
# # particleState.supports = h_optimal

# # particles.densities = rho_optimal
# # particles.supports = h_optimal

# print(f' Support After Optimization: Min: {particles.supports.min().item()}, Max: {particles.supports.max().item()}, Mean: {particles.supports.mean().item()}')

# P_initial = torch.zeros_like(particles.densities)
# u = 1 / (gamma - 1) * (P_initial / rho_optimal)
# # A_, u_, P_, c_s = idealGasEOS(A = None, u = None, P = P_initial, rho = rho_optimal, gamma = gamma)
# A_, u_, P_, c_s = idealGasEOS(A = None, u = u, P = None, rho = rho_optimal, gamma = gamma)

# internalEnergy = u_ 
# kineticEnergy = torch.linalg.norm(torch.zeros_like(particles.positions), dim = -1) **2/ 2
# totalEnergy = (internalEnergy + kineticEnergy) * particles.masses

# simulationState_ = SimulationState(
#     positions = particles.positions,
#     supports = particles.supports,
#     masses = particles.masses,
#     densities = particles.densities,        
#     velocities = torch.zeros_like(particles.positions),

#     kinds = torch.zeros_like(particles_.positions[:,0], dtype = torch.int32),
#     materials = torch.zeros_like(particles_.positions[:,0], dtype = torch.int32),
#     UIDs = torch.arange(particles_.positions.shape[0], device = device, dtype = torch.int32),
#     UIDcounter = particles.positions.shape[0],
    
#     internalEnergies = u_,
#     totalEnergies = totalEnergy,
#     entropies = A_,
#     pressures = P_,
#     soundspeeds = c_s,

#     alphas = torch.ones_like(particles.densities),
#     alpha0s = torch.ones_like(particles.densities),
#     divergence=torch.zeros_like(particles.densities),
# )
    


# adjacency = buildVerletList(simulationState_, 
#                             domain = config.domain,
#                             verletScale = 2**(1/config.dim), supportMode = config.supportMode)

# compressibleSystem = SimulationSystem(
#     state=simulationState_, 
#     adjacency = adjacency, 
#     domain = config.domain)

# # config.domain.min[0] = -L/2/aspect
# # config.domain.max[0] = L/2/aspect
# # config.domain.min[1] = 0 - 2*band * dx
# # config.domain.max[1] = L + 2*band * dx

In [8]:
particles = compressibleSystem.state

In [9]:
# rho_b = rho_low
# rho_t = rho_high
# delta = 0.0025
# g = 1/2
# gamma = compressibleSPHConfig.gamma
# P_0 = rho_t / gamma
# print(gamma)


In [10]:
def rayleighTaylor_rho(positions, rho_b, rho_t, delta):
    # print('Enforcing Rayleigh-Taylor density for ', positions.shape[0], ' particles')
    y = positions[:,1]
    rho_y = rho_b + (rho_t - rho_b) * (1 + torch.exp(-(y - 0.5) / delta))**(-1)
    return rho_y
def rayleighTaylor_u(positions, rho_b, rho_t, delta, g, gamma):
    y = positions[:,1]
    rho_y = rayleighTaylor_rho(positions, rho_b, rho_t, delta)
    P_0 = rho_t / gamma
    P = P_0 - g * rho_y * (y - 1/2)
    u = P / rho_y / (gamma - 1)
    return u

x = particles.positions[:,0]
y = particles.positions[:,1]

# rho_y = rho_b + (rho_t - rho_b) * (1 + torch.exp(-(y - 0.5) / delta))**(-1)
rho_y = rayleighTaylor_rho(particles.positions, rho_b, rho_t, delta)

delta_y = delta * 5
v_y = delta_y * (1 + torch.cos(8 * np.pi * (x + 0.25))) * (1 + torch.cos(5 * np.pi * (y - 0.5)))
v_y[~((y >= 0.3) & (y <= 0.7))] = 0

P = P_0 - g * rho_y * (y - 1/2)

Pinitial = P
rhoInitial = rho_y
vInitial = torch.stack([torch.zeros_like(v_y), v_y], dim=-1)



In [11]:
def buffer_sdf(positions):
    dist = torch.zeros_like(positions[:,0])

    maskA = positions[:,1] < 0
    maskB = positions[:,1] > L
    maskC = torch.logical_and(positions[:,1] >= 0, positions[:,1] <= L/2)
    maskD = torch.logical_and(positions[:,1] > L/2, positions[:,1] <= L)

    dist[maskA] = positions[maskA,1]
    dist[maskB] = L - positions[maskB,1]
    dist[maskC] = positions[maskC,1]
    dist[maskD] = L - positions[maskD,1]
    return dist

def buffer_sdf_gradient(positions):
    dist = torch.zeros_like(positions)

    maskA = positions[:,1] < 0
    maskB = positions[:,1] > L
    maskC = torch.logical_and(positions[:,1] >= 0, positions[:,1] <= L/2)
    maskD = torch.logical_and(positions[:,1] > L/2, positions[:,1] <= L)

    dist[maskA,1] = 1
    dist[maskB,1] = -1
    dist[maskC,1] = -1
    dist[maskD,1] = 1
    return dist

In [12]:

def RayleighTaylorVelocity(positions):
    return torch.zeros_like(positions)

def RayleighTaylorAcceleration(positions):
    return torch.zeros_like(positions)

def RayleighTaylorDensity(positions):
    return rayleighTaylor_rho(positions, rho_b, rho_t, delta)

def RayleighTaylorInternalEnergy(positions):
    u = rayleighTaylor_u(positions, rho_b, rho_t, delta, g, gamma)
    return u


rayleighTaylorBC = BoundaryCondition(
    type = BoundaryConditionType.dynamic,
    sdf = lambda x: (buffer_sdf(x), buffer_sdf_gradient(x)),
    dirichletFunctions = {
        'velocities': lambda state, cfg, schemeCfg, positions, d, n, t, dt: RayleighTaylorVelocity(positions),
        'densities': lambda state, cfg, schemeCfg, positions, d, n, t, dt: RayleighTaylorDensity(positions),
        'internalEnergies': lambda state, cfg, schemeCfg, positions, d, n, t, dt: RayleighTaylorInternalEnergy(positions),
    },
        updateFunctions = {
            'dvdt': lambda state, cfg, schemeCfg, positions, d, n, t, dt: RayleighTaylorAcceleration(positions),
            'dxdt': lambda state, cfg, schemeCfg, positions, d, n, t, dt: RayleighTaylorVelocity(positions),
        }
)
schemeConfig.boundaryConditions.clear()
schemeConfig.boundaryConditions.append(rayleighTaylorBC)

In [13]:

def gravityForcing(state, cfg, schemeCfg, positions, d, n, t, dt):
    masses = state.masses
    dvdt = torch.zeros_like(positions)
    dvdt[:, 1] = -g 
    return dvdt * masses.view(-1,1)

gravityBC = BoundaryCondition(
    type = BoundaryConditionType.dynamic,
    sdf = lambda x: (-1 * torch.ones_like(x[:,0]), torch.tensor([0,-1], device = x.device, dtype = x.dtype).expand(x.shape[0], -1)),
    forcingFunctions=[
        gravityForcing,
    ]
)
schemeConfig.boundaryConditions.clear()
schemeConfig.boundaryConditions.append(rayleighTaylorBC)
schemeConfig.boundaryConditions.append(gravityBC)

enforceDirichlet(compressibleSystem, compressibleSystem.t, config.dt, config, schemeConfig)

In [14]:

compressibleSystem.state.masses = dx**config.dim / aspect**dim * rhoInitial

compressibleSPHConfigAdaptiveH = CompressibleSPHConfig(
    adaptiveSupportIterations=16,
    adaptiveSupportThreshold=1e-3,
    adaptiveSupportScheme=AdaptiveSupportScheme.Owen,
)

rho_optimal, h_optimal, adjacency, rhos_iter, supports_iter = evaluateOptimalSupport(compressibleSystem.state, config, supportScheme = SupportScheme.Gather, compParams = compressibleSPHConfigAdaptiveH)

compressibleSystem.state.supports = h_optimal
compressibleSystem.state.densities = rho_optimal

Module compressibleSPH.modules.adaptiveSupport.wp_psi 8875fe6 load on device 'cuda:0' took 5.55 ms  (cached)
Module sphWarpCore.radiusSearch.wp_compactHash e67fccd load on device 'cuda:0' took 4.78 ms  (cached)
Module compressibleSPH.modules.adaptiveSupport.wp_psi0 37e5819 load on device 'cuda:0' took 6.18 ms  (cached)
Module sphWarpCore.operations.wp_density f0357bf load on device 'cuda:0' took 6.32 ms  (cached)


In [19]:
A_, u_, P_, c_s = idealGasEOS(A = None, u = None, P = Pinitial, rho = compressibleSystem.state.densities, gamma = gamma)
# v_initial = torch.zeros_like(particles_l.positions)

internalEnergy = u_ 
kineticEnergy = torch.linalg.norm(vInitial, dim = -1) **2/ 2
totalEnergy = (internalEnergy + kineticEnergy) * compressibleSystem.state.masses

compressibleSystem.state.internalEnergies = u_
compressibleSystem.state.totalEnergies = totalEnergy
compressibleSystem.state.pressures = P_
compressibleSystem.state.soundspeeds = c_s
compressibleSystem.state.velocities = vInitial
# compressibleSystem.state.densities = rhoInitial


from compressibleSPH.modules.timestep.compressible import computeTimestep

# config.dt = computeTimestep(compressibleSystem, config, schemeConfig, dt = None)
config.dt = computeTimestep(compressibleSystem, config, schemeConfig, dt = None) * 2/3

In [20]:
runningState = compressibleSystem.initializeNewState()

kineticEnergy = 0.5 * (torch.linalg.norm(runningState.state.velocities, dim = -1) **2 * runningState.state.masses).sum()
thermalEnergy = (runningState.state.internalEnergies * runningState.state.masses).sum()
totalEnergy = kineticEnergy + thermalEnergy

In [21]:
caseName = '13-Rayleigh_Taylor'
exportPath = prepExport(f'{caseName}', config, schemeConfig, scheme, export_fn)
exportSimulationSystem(exportPath, 'initialState', scheme, compressibleSystem, exportAdjacency = False, stages = None, exportStagesAdjacency = False, extraData = dict({
    'kineticEnergy': kineticEnergy,
    'thermalEnergy': thermalEnergy,
    'totalEnergy': totalEnergy,
    'frame_num': 0,
}, **extraData))


In [22]:
from warpPlot import visualize, PlottingOptions, PlotScaling, GridVisualization, UniformColorMap, DivergingColorMap, Mapping, CyclicColorMap
markerSize = 2
plotter = visualize(
    particleState = runningState.state,
    domain = config.domain,
    quantities = {
        "A": runningState.state.velocities,
        "B": runningState.state.densities,
    },
    plotOptions = {
        "A": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            # colorMap = CyclicColorMap.twilight,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "Velocity",
            mapping = Mapping.L2,
            # plottingOperation = OperationProperties(
            #     kernel = config.kernel,
            #     operation = WarpOperation.Curl,
            #     supportMode = SupportScheme.Gather,
            #     gradientMode = config.gradientMode,
            # ),
            gridVisualization = GridVisualization(
                resolution = 1024,
            ),
            # vMin=1e-10
        ),
        "B": PlottingOptions(
            colorMap = DivergingColorMap.RdBu,
            flipColorMap = True,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "density",
            gridVisualization = GridVisualization(
                resolution = 1024,
            ),
            vMin=0.95,
            vMax=2.05
        ),
    },
    figTitle = "Wave Equation Example",
    mosaic = 'AB',
    figsize= (8,7),
    backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)
imagePath = f'{exportPath}/images'
os.makedirs(imagePath, exist_ok = True)
plotter.export(f'{imagePath}/frame_00000.png', dpi = 300)

# if args.exportImages:

RFBOutputContext()

In [23]:
# config.dt = 2.5e-3
t_limit = 10.0
nSteps = int(t_limit / config.dt)

print(f"Running with dt: {config.dt}, which gives nSteps: {nSteps}")
# nSteps = 256

runningState = compressibleSystem.initializeNewState()

trajectory = []

priorStep = None
for i in (tq := tqdm(range(nSteps), leave = True)):
    begin = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    begin.record()
    result = integrator.function(
        state = runningState,
        f = fn,
        dt = config.dt,  
        config = config,
        compParams = schemeConfig,
        verbose = False,
        # priorStep = priorStep
    )
    end.record()
    torch.cuda.synchronize()
    priorStep = result.stages[-1]
    timing = begin.elapsed_time(end)

    runningState = result.state
    kineticEnergy = 0.5 * (torch.linalg.norm(runningState.state.velocities, dim = -1) **2 * runningState.state.masses).sum()
    thermalEnergy = (runningState.state.internalEnergies * runningState.state.masses).sum()
    totalEnergy = kineticEnergy + thermalEnergy

    trajectory.append(
        (i, (i+1)*config.dt, totalEnergy.item(), kineticEnergy.item(), thermalEnergy.item(), timing)
,     )


    if i % 10 == 0 and i > 0:
        plotter.updateQuantities(
            {
                "A": runningState.state.velocities,
                "B": runningState.state.densities,
            },
            newParticleState = runningState.state,
        )
        plotter.export(f'{imagePath}/frame_{i:05d}.png', dpi = 300)
        
    if i % 500 == 0:
        exportSimulationSystem(exportPath, f'state_{i:04d}', scheme, runningState, exportAdjacency = False, stages = result.stages, exportStagesAdjacency = True, extraData = dict(**extraData, **{
            'kineticEnergy': kineticEnergy,
            'thermalEnergy': thermalEnergy,
            'totalEnergy': totalEnergy,
            'frame_num': i,
        }))

        
    maxVel = torch.linalg.norm(runningState.state.velocities, dim = -1).max()
    tq.set_description(f"Step {i+1}/{nSteps}, time: {(i+1)*config.dt:8.4g}/{t_limit:8.4g}, TE: {totalEnergy:.3g}, KE: {kineticEnergy:.3g}, IE: {thermalEnergy:.3g} | max vel: {maxVel:.3g} | iter time: {timing:.3f} ms")
    # t = {runningState.t:2f}, dt = {config.dt:.3g}, ptcls = {len(runningState.state.positions)}\nTotal Energy: {totalEnergy:.3g}, Kinetic Energy: {kineticEnergy:.3g}, Thermal Energy: {thermalEnergy:.3g}'
    # break

Running with dt: 0.0005204380334665378, which gives nSteps: 19214


  0%|          | 0/19214 [00:00<?, ?it/s]

Module sphWarpCore.crk.crk_volume 793e787 load on device 'cuda:0' took 5.03 ms  (cached)
Module sphWarpCore.crk.crk_moments 89bc989 load on device 'cuda:0' took 7.27 ms  (cached)
Module sphWarpCore.crk.crk_density 06a7669 load on device 'cuda:0' took 6.35 ms  (cached)
Module sphWarpCore.operations.wp_gradient a50c471 load on device 'cuda:0' took 9.48 ms  (cached)
Module compressibleSPH.modules.crk.accel 9056830 load on device 'cuda:0' took 7.36 ms  (cached)
Module compressibleSPH.modules.crk.dudt 7e9ded3 load on device 'cuda:0' took 7.04 ms  (cached)
Module compressibleSPH.modules.compSPH.balance 8be6a68 load on device 'cuda:0' took 5.14 ms  (cached)


In [24]:
exportSimulationSystem(exportPath, f'finalState', scheme, runningState, exportAdjacency = False, stages = result.stages, exportStagesAdjacency = True, extraData = dict(**extraData, **{
    'kineticEnergy': kineticEnergy,
    'thermalEnergy': thermalEnergy,
    'totalEnergy': totalEnergy,
    'frame_num': i,
}))

In [25]:
ffmpeg_cmd = "ffmpeg -y -loglevel error -hide_banner -framerate 50 -f image2 -pattern_type glob -i 'frame_*.png' -c:v libx264 -pix_fmt yuv420p -b:v 10M output.mp4"
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)
ffmpeg_cmd = 'ffmpeg -y -loglevel error -hide_banner -i output.mp4  -vf "fps=50,scale=540:-1:flags=lanczos,palettegen" palette.png'
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)
ffmpeg_cmd = 'ffmpeg -y -loglevel error -hide_banner -i output.mp4 -i palette.png -filter_complex "fps=25,scale=540:-1:flags=lanczos[x];[x][1:v]paletteuse" out.gif'
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)

# now copy the output.mp4 and out.gif to the parent directory for easier access
shutil.copy(f'{imagePath}/output.mp4', f'{exportPath}/output.mp4')
shutil.copy(f'{imagePath}/out.gif', f'{exportPath}/out.gif');

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(11,10), squeeze=False)

sc = axis[0,0].scatter(particles.positions[:,0].cpu(), particles.positions[:,1].cpu(), c = rho_y.cpu(), s = 1, cmap = 'viridis')
fig.colorbar(sc, ax = axis[0,0])

sc = axis[0,1].scatter(particles.positions[:,0].cpu(), particles.positions[:,1].cpu(), c = P.cpu(), s = 1, cmap = 'viridis')
fig.colorbar(sc, ax = axis[0,1])

for ax in axis.flatten():
    ax.set_aspect('equal')
    ax.set_xlim(config.domain.min[0].item(), config.domain.max[0].item())
    ax.set_ylim(config.domain.min[1].item(), config.domain.max[1].item())
    ax.set_xlabel('x')
    ax.set_ylabel('y')

fig.tight_layout()

In [ ]:
def buffer_sdf(positions):
    dist = torch.zeros_like(positions[:,0])

    maskA = positions[:,1] < 0
    maskB = positions[:,1] > L
    maskC = torch.logical_and(positions[:,1] >= 0, positions[:,1] <= L/2)
    maskD = torch.logical_and(positions[:,1] > L/2, positions[:,1] <= L)

    dist[maskA] = positions[maskA,1]
    dist[maskB] = L - positions[maskB,1]
    dist[maskC] = positions[maskC,1]
    dist[maskD] = L - positions[maskD,1]
    return dist

def buffer_sdf_gradient(positions):
    dist = torch.zeros_like(positions)

    maskA = positions[:,1] < 0
    maskB = positions[:,1] > L
    maskC = torch.logical_and(positions[:,1] >= 0, positions[:,1] <= L/2)
    maskD = torch.logical_and(positions[:,1] > L/2, positions[:,1] <= L)

    dist[maskA,1] = 1
    dist[maskB,1] = -1
    dist[maskC,1] = -1
    dist[maskD,1] = 1
    return dist


sdf = buffer_sdf(particles.positions)
sdf_gradient = buffer_sdf_gradient(particles.positions)

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(11,10), squeeze=False)
bandWidth = band * dx
sc = axis[0,0].scatter(particles.positions[:,0].cpu(), particles.positions[:,1].cpu(), c = sdf.cpu(), s = 1, cmap = 'RdBu_r', vmin = -bandWidth, vmax = bandWidth)
fig.colorbar(sc, ax = axis[0,0])

sc = axis[0,1].scatter(particles.positions[:,0].cpu(), particles.positions[:,1].cpu(), c = sdf_gradient[:,1].cpu(), s = 1, cmap = 'viridis')
fig.colorbar(sc, ax = axis[0,1])

for ax in axis.flatten():
    ax.set_aspect('equal')
    ax.set_xlim(config.domain.min[0].item(), config.domain.max[0].item())
    ax.set_ylim(config.domain.min[1].item(), config.domain.max[1].item())
    ax.set_xlabel('x')
    ax.set_ylabel('y')

fig.tight_layout()

In [ ]:
from compressibleSPH.boundaryConditions import *


In [ ]:
forcing = computeForcing(compressibleSystem.state, compressibleSystem.t, config.dt, config, compressibleSPHConfig)
print(forcing / particles.masses.view(-1,1))

In [ ]:
config.dx**config.dim / aspect**dim

In [ ]:
Pinitial = Pinitial
rhoInitial = rho_y


A_, u_, P_, c_s = idealGasEOS(A = None, u = None, P = Pinitial, rho = rhoInitial, gamma = gamma)
# v_initial = torch.zeros_like(particles_l.positions)

internalEnergy = u_ 
kineticEnergy = torch.linalg.norm(vInitial, dim = -1) **2/ 2
totalEnergy = (internalEnergy + kineticEnergy) * particles.masses

particles.internalEnergies = u_
particles.totalEnergies = totalEnergy
particles.pressures = P_
particles.soundspeeds = c_s
particles.velocities = vInitial
particles.densities = rhoInitial
particles.masses = config.dx**config.dim / aspect**dim * rhoInitial

compressibleSystem = SimulationSystem(
    state=particles, 
    adjacency = adjacency, 
    domain = config.domain)

In [ ]:

# rho_optimal, h_optimal, adjacency, rhos_iter, supports_iter = evaluateOptimalSupport(particles, config, supportScheme = SupportScheme.Gather, compParams = compressibleSPHConfig)

# compressibleSystem.state.supports = h_optimal


In [ ]:

adjacency = buildVerletList(
    compressibleSystem.state, 
    config.domain, verletScale = 1.0, supportMode = SupportScheme.SuperSymmetric,
    priorNeighborhood = None,
    verbose = False)

apparentVolume, compressibleSystem.state.densities, crkState = computeCRKFactors(compressibleSystem.state, config.domain, config.kernel, adjacency = adjacency)

In [ ]:
enforceDirichlet(compressibleSystem.state, compressibleSystem.t, config.dt, config, compressibleSPHConfig)

In [ ]:
from warpPlot import visualize, PlottingOptions, PlotScaling, GridVisualization, UniformColorMap, DivergingColorMap, Mapping
markerSize = 12.5
plotter = visualize(
    particleState = compressibleSystem.state,
    domain = config.domain,
    quantities = {
        "A": compressibleSystem.state.densities,
        "B": h_optimal,
    },
    plotOptions = {
        "A": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "Densities",
            # mapping = Mapping.L2,
            # plottingOperation = OperationProperties(
            #     kernel = config.kernel,
            #     operation = WarpOperation.Curl,
            #     supportMode = SupportScheme.Gather,
            #     gradientMode = config.gradientMode,
            # ),
            # gridVisualization = GridVisualization(
            #     resolution = 256,
            # ),
            # vMin=1e-10
        ),
        "B": PlottingOptions(
            colorMap = DivergingColorMap.RdBu,
            flipColorMap = True,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "Support Radius",
            # gridVisualization = GridVisualization(
            #     resolution = 1024,
            # ),
            # vMin=0.95,
            # vMax=2.05
        ),
    },
    figTitle = "Wave Equation Example",
    mosaic = 'AB',
    figsize= (8,7),
    backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)

# if args.exportImages:
#     plotter.export(f'output/{folderName}/frame_00000.png', dpi = args.figureDpi)

In [ ]:
from warpPlot import *

In [ ]:
KE = (compressibleSystem.state.masses * torch.linalg.norm(compressibleSystem.state.velocities, dim = -1)**2 / 2).sum()
IE = (compressibleSystem.state.masses * compressibleSystem.state.internalEnergies).sum()
TE = (compressibleSystem.state.masses * (compressibleSystem.state.internalEnergies + torch.linalg.norm(compressibleSystem.state.velocities, dim = -1)**
2)).sum()
print(f"Initial KE: {KE}, IE: {IE}, TE: {TE}, TE+PE: {TE}")

nonzeroEnergyMask = compressibleSystem.state.internalEnergies > 1e-8
print(f"Nonzero internal energy particles: {nonzeroEnergyMask.sum()} / {compressibleSystem.state.internalEnergies.shape[0]}")

In [ ]:
from compressibleSPH.schemes.compSPH import compSPH_step
from compressibleSPH.modules.timestep.compressible import computeTimestep
config.dt = computeTimestep(compressibleSystem, config, compressibleSPHConfig, dt = config.dt) * 2/3
print(f"Initial timestep: {config.dt}")

In [ ]:
runningState = compressibleSystem.initializeNewState()

In [ ]:
compressibleSPHConfig.adaptiveSupportScheme = AdaptiveSupportScheme.Owen
compressibleSPHConfig.viscositySwitchParams.scheme = ViscositySwitch.CullenHopkins
# schemeConfig.diffusionParams.C_q = 1.0



In [ ]:
from warpPlot import visualize, PlottingOptions, PlotScaling, GridVisualization, UniformColorMap, DivergingColorMap, Mapping, CyclicColorMap
markerSize = 2
plotter = visualize(
    particleState = runningState.state,
    domain = config.domain,
    quantities = {
        "A": runningState.state.velocities,
        "B": runningState.state.densities,
    },
    plotOptions = {
        "A": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            # colorMap = CyclicColorMap.twilight,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "Velocity",
            mapping = Mapping.L2,
            # plottingOperation = OperationProperties(
            #     kernel = config.kernel,
            #     operation = WarpOperation.Curl,
            #     supportMode = SupportScheme.Gather,
            #     gradientMode = config.gradientMode,
            # ),
            gridVisualization = GridVisualization(
                resolution = 1024,
            ),
            # vMin=1e-10
        ),
        "B": PlottingOptions(
            colorMap = DivergingColorMap.RdBu,
            flipColorMap = True,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "density",
            gridVisualization = GridVisualization(
                resolution = 1024,
            ),
            # vMin=0.95,
            # vMax=2.05
        ),
    },
    figTitle = "Wave Equation Example",
    mosaic = 'AB',
    figsize= (8,7),
    backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)

# if args.exportImages:

In [ ]:
folderName = f"rayleighTaylor_{getCurrentTimestamp()}"
os.makedirs(f'output/{folderName}', exist_ok = True)
plotter.export(f'output/{folderName}/frame_00000.png', dpi = 200)

In [ ]:
t_limit = 1.0
# config.dt = 1e-4
nSteps = int(t_limit / config.dt)
integrator = getIntegrator(config.integrationScheme)


print(f"Running with dt: {config.dt}, which gives nSteps: {nSteps}")
# nSteps = 54
print(nSteps//25)

In [ ]:
# os.makedirs('images/kevinHH_256_2', exist_ok = True)

t_limit = 10.0
# t_limit = 0.1
# config.dt = 1e-4
nSteps = int(t_limit / config.dt)
integrator = getIntegrator(config.integrationScheme)


print(f"Running with dt: {config.dt}, which gives nSteps: {nSteps}")
# nSteps = 54
 
trajectory = []
runningState = compressibleSystem.initializeNewState()
priorStep = None
for i in (tq := tqdm(range(nSteps), leave = True)):
    begin = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    begin.record()
    # print(f"Step {i+1}/{nSteps}, time: {runningState.t:.4f}")
    result = integrator.function(
        state = runningState,
        f = fn,
        dt = config.dt,
        config = config,
        compParams = compressibleSPHConfig,
        verbose = False,
        priorStep = priorStep
    )
    end.record()
    torch.cuda.synchronize()
    elapsed_time_ms = begin.elapsed_time(end)
    densityRatio = runningState.state.densities / result.state.state.densities
    
    # if (densityRatio < 0.7).any() or (densityRatio > 1.5).any():
    #     print(f"Warning: Density ratio out of bounds at step {i}, min: {densityRatio.min().item()}, max: {densityRatio.max().item()}")
    #     break

    runningState = result.state
    priorStep = result.stages[-1]


    kineticEnergy_ = 0.5 * (torch.linalg.norm(runningState.state.velocities, dim = -1) **2 * runningState.state.masses).sum()
    thermalEnergy_ = (runningState.state.internalEnergies * runningState.state.masses).sum()
    totalEnergy = kineticEnergy_ + thermalEnergy_

    tq.set_description(f"t: {runningState.t:.4f}, KE: {kineticEnergy_:.4f}, TE: {thermalEnergy_:.4f}, TE+KE: {totalEnergy:.4f}")
        
    trajectory.append((
        i,
        runningState.t,
        elapsed_time_ms,
        kineticEnergy_.cpu().item(),
        thermalEnergy_.cpu().item(),
        totalEnergy.cpu().item(),
    ))
    if (i % 10 == 0 and i > 0) or i == nSteps - 1:
        plotter.updateQuantities(
            {
                "A": runningState.state.velocities,
                "B": runningState.state.densities,
            },
            newParticleState = runningState.state,
        )
        plotter.export(f'output/{folderName}/frame_{i:05d}.png', dpi = 300)
        # for ax in axis.flatten():
        #     ax.clear()
        # plotSod_(fig, axis, runningState.state, config, schemeConfig, config.domain, gamma, leftState, rightState, plotReference = True, plotLabels = False, scatter = True, t_ = runningState.t)
        # fig.canvas.draw()
        # fig.canvas.flush_events()
    # print(f' Supports: Min: {runningState.state.supports.min().item()}, Max: {runningState.state.supports.max().item()}, Mean: {runningState.state.supports.mean().item()}')

In [ ]:
result = integrator.function(
    state = runningState,
    f = fn,
    dt = config.dt,
    config = config,
    compParams = compressibleSPHConfig,
    verbose = False,
    priorStep = priorStep
)

densityRatio = runningState.state.densities / result.state.state.densities

if (densityRatio < 0.7).any() or (densityRatio > 1.5).any():
    print(f"Warning: Density ratio out of bounds at step {i}, min: {densityRatio.min().item()}, max: {densityRatio.max().item()}")

print(runningState.state.densities.max(), runningState.state.densities.min())
print(result.state.state.densities.max(), result.state.state.densities.min())

In [ ]:
from integrators.euler import updateStateEuler

halfState = runningState.initializeNewState()
halfState = updateStateEuler(halfState, priorStep.update, config.dt * 0.5, False)

In [ ]:
# halfState.adjacency = None

In [ ]:
config.verletScale

In [ ]:
stepResult = fn(halfState, config.dt * 0.5, config, compressibleSPHConfig)

In [ ]:
nnrsPrev = runningState.adjacency.numNeighbors
print(f'Number of neighbors: min {nnrsPrev.min().item()}, max {nnrsPrev.max().item()}, mean {nnrsPrev.to(torch.float32).mean().item()}')
nnrsCurr = stepResult[1].numNeighbors
print(f'Number of neighbors: min {nnrsCurr.min().item()}, max {nnrsCurr.max().item()}, mean {nnrsCurr.to(torch.float32).mean().item()}')

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(11,10), squeeze=False)
sc = axis[0,0].scatter(runningState.state.positions[:,0].cpu(), runningState.state.positions[:,1].cpu(), c = nnrsPrev.cpu(), s = 1, cmap = 'viridis')
fig.colorbar(sc, ax = axis[0,0])
sc = axis[0,1].scatter(stepResult[2].positions[:,0].cpu(), stepResult[2].positions[:,1].cpu(), c = nnrsCurr.cpu(), s = 1, cmap = 'viridis')
fig.colorbar(sc, ax = axis[0,1])
for ax in axis.flatten():
    ax.set_aspect('equal')
    ax.set_xlim(config.domain.min[0].item(), config.domain.max[0].item())
    ax.set_ylim(config.domain.min[1].item(), config.domain.max[1].item())
    ax.set_xlabel('x')
    ax.set_ylabel('y')
fig.tight_layout()

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(11,10), squeeze=False)
sc = axis[0,0].scatter(runningState.state.positions[:,0].cpu(), runningState.state.positions[:,1].cpu(), c = runningState.state.densities.cpu(), s = 1, cmap = 'viridis')
fig.colorbar(sc, ax = axis[0,0])
sc = axis[0,1].scatter(stepResult[2].positions[:,0].cpu(), stepResult[2].positions[:,1].cpu(), c = stepResult[2].densities.cpu(), s = 1, cmap = 'viridis')
fig.colorbar(sc, ax = axis[0,1])
for ax in axis.flatten():
    ax.set_aspect('equal')
    ax.set_xlim(config.domain.min[0].item(), config.domain.max[0].item())
    ax.set_ylim(config.domain.min[1].item(), config.domain.max[1].item())
    ax.set_xlabel('x')
    ax.set_ylabel('y')
fig.tight_layout()

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(11,10), squeeze=False)
sc = axis[0,0].scatter(runningState.state.positions[:,0].cpu(), runningState.state.positions[:,1].cpu(), c = runningState.state.supports.cpu(), s = 1, cmap = 'viridis')
fig.colorbar(sc, ax = axis[0,0])
sc = axis[0,1].scatter(stepResult[2].positions[:,0].cpu(), stepResult[2].positions[:,1].cpu(), c = stepResult[2].supports.cpu(), s = 1, cmap = 'viridis')
fig.colorbar(sc, ax = axis[0,1])
for ax in axis.flatten():
    ax.set_aspect('equal')
    ax.set_xlim(config.domain.min[0].item(), config.domain.max[0].item())
    ax.set_ylim(config.domain.min[1].item(), config.domain.max[1].item())
    ax.set_xlabel('x')
    ax.set_ylabel('y')
fig.tight_layout()

In [ ]:
import copy
queryPositions = halfState.state.positions
referencePositions = halfState.state.positions
querySupports = halfState.state.supports
referenceSupports = halfState.state.supports
domain = copy.deepcopy(config.domain)
domain.min = domain.min
domain.max = domain.max

verletScale = 1.0
mode = SupportScheme.Scatter

adjacency, hmap = radiusSearchCompactHashMap_(
    queryPositions, referencePositions,
    querySupports * verletScale, referenceSupports * verletScale,
    domain.periodic, domain, mode, hashMapLength=queryPositions.shape[0] + 1, returnCompactHashMap = True
)

print(adjacency.numNeighbors.min(), adjacency.numNeighbors.max(), adjacency.numNeighbors.to(torch.float).mean())

In [ ]:

print(torch.sum(adjacency.i == adjacency.j), queryPositions.shape[0])

In [ ]:
print(hmap.hashTable)
print(hmap.sortedCellTable.shape)

In [ ]:

queryPositions = halfState.state.positions.to(torch.float64)
referencePositions = halfState.state.positions.to(torch.float64)
querySupports = halfState.state.supports.to(torch.float64)
referenceSupports = halfState.state.supports.to(torch.float64)
domain = copy.deepcopy(config.domain)
domain.min = domain.min.to(torch.float64)
domain.max = domain.max.to(torch.float64)

In [ ]:
domainDescription = config.domain
periodicity = domainDescription.periodic

mode_uint = supportSchemeToUint(mode)
# mode_map = {'gather': 1, 'scatter': 2, 'symmetric': 3, 'superSymmetric': 4, ''}
# mode_uint = mode_map.get(mode, 0)
# if mode_uint == 0:
    # raise ValueError(f"Invalid mode: {mode}. Supported modes are: {list(mode_map.keys())}")
    
minDomain = domainDescription.min if domainDescription.min is not None else None
maxDomain = domainDescription.max if domainDescription.max is not None else None
hMax = computeGridSupport(querySupports, referenceSupports, mode)
minD, maxD = getDomainExtents(referencePositions, minDomain, maxDomain)
x = torch.vstack([component if not periodic else torch.remainder(component - minD[i], maxD[i] - minD[i]) + minD[i] for i, (component, periodic) in enumerate(zip(referencePositions.mT, periodicity))]).mT
y = torch.vstack([component if not periodic else torch.remainder(component - minD[i], maxD[i] - minD[i]) + minD[i] for i, (component, periodic) in enumerate(zip(queryPositions.mT, periodicity))]).mT

sortedLinear, sortIndex, numCells, qMin, qMax, hCell = sortReferenceParticles(x, hMax, minD, maxD)


sortedPositions = x[sortIndex,:]

cellIndices, cellCounters = torch.unique_consecutive(sortedLinear, return_counts=True, return_inverse=False)
cellCounters = cellCounters.to(torch.int32)
# Needs to zero padded for the indexing to work properly as the 0th cell is valid and cumsum doesn't include the first element

cumCell = torch.hstack((torch.tensor([0], device = cellIndices.device, dtype=cellCounters.dtype),torch.cumsum(cellCounters,dim=0)))[:-1].to(torch.int32)

sortedIndices = torch.floor((sortedPositions - qMin) / to_numpy(hCell)).to(torch.int32)
for d in range(sortedIndices.shape[1]):
    sortedIndices[:, d] = torch.clamp(sortedIndices[:, d], 0, int(numCells[d].item()) - 1)
cellGridIndices = sortedIndices[cumCell,:]
cellTable = torch.stack((cellIndices, cumCell, cellCounters), dim = 1)

warpDevice = castTorchToWarp(queryPositions).device
torchDevice = queryPositions.device

cellGridIndices_warp = castTorchToWarp(cellGridIndices)
hashedIndices_warp = wp.zeros(sortedIndices.shape[0], dtype=wp.uint32, device=warpDevice)
wp.launch(hashCells, dim=sortedIndices.shape[0], inputs=[sortedIndices, wp.uint32(2), hashedIndices_warp], device=warpDevice)
hashedIndices = wp.to_torch(hashedIndices_warp).to(torch.int32)

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(11,10), squeeze=False)

q = sortedLinear
qUnique = torch.unique(q)
colors = torch.rand((qUnique.shape[0], 3), device = queryPositions.device)
c = colors[torch.searchsorted(qUnique, q)]

s = 1

sc = axis[0,0].scatter(sortedPositions[:,0].cpu(), sortedPositions[:,1].cpu(), c = c.cpu(), s = s, cmap = 'viridis')
# fig.colorbar(sc, ax = axis[0,0])

q = hashedIndices
qUnique = torch.unique(q)
colors = torch.rand((qUnique.shape[0], 3), device = queryPositions.device)
c = colors[torch.searchsorted(qUnique, q)]

sc = axis[0,1].scatter(sortedPositions[:,0].cpu(), sortedPositions[:,1].cpu(), c = c.cpu(), s = s, cmap = 'viridis')
# fig.colorbar(sc, ax = axis[0,1])
for ax in axis.flatten():
    ax.set_aspect('equal')
    ax.set_xlim(config.domain.min[0].item(), config.domain.max[0].item())
    ax.set_ylim(config.domain.min[1].item(), config.domain.max[1].item())
    ax.set_xlabel('x')
    ax.set_ylabel('y')

In [ ]:
ffmpeg -framerate 50 -f image2 -pattern_type glob -i 'frame_*.png' -c:v libx264 -pix_fmt yuv420p -b:v 10M output.mp4
ffmpeg -i output.mp4  -vf "fps=50,scale=540:-1:flags=lanczos,palettegen" palette.png
ffmpeg -i output.mp4 -i palette.png -filter_complex "fps=25,scale=540:-1:flags=lanczos[x];[x][1:v]paletteuse" out.gif

In [ ]:
cState = runningState.initializeNewState()
currentState = cState.state


adjacency = buildVerletList(
    currentState, 
    config.domain, verletScale = 1.0, supportMode = SupportScheme.SuperSymmetric,
    priorNeighborhood = None,
    verbose = False)

apparentVolume, currentState.densities, crkState = computeCRKFactors(currentState, config.domain, config.kernel, adjacency = adjacency)


In [ ]:
def limiterVL(x):
    # if x <= 0.0:
        # return 0.0
    # x = wp.min(x, scalar_t(1.0e6))
    # vL = 2.0 / (1.0 + x)
    # return x * vL*vL

    return (x + torch.abs(x)) / (1.0 + torch.abs(x))

xx = torch.linspace(0, 1, 1000)
yy = limiterVL(xx)

fig, ax = plt.subplots()
ax.plot(xx.cpu(), yy.cpu())
ax.set_title("Van Leer Limiter")
ax.set_xlabel("x")
ax.set_ylabel("limiterVL(x)")
plt.grid()


In [ ]:
velocityGradient = warpOperation(
    currentState,
    OperationProperties(
        kernel = config.kernel,
        operation = WarpOperation.Gradient,
        supportMode = SupportScheme.Scatter, # E.3
        gradientMode = GradientScheme.Difference, # E.3
    ),
    queryValues = currentState.velocities,
    domain = config.domain,
    adjacency = adjacency,
    # queryVolumes = apparentVolume,
    # crkState= crkState,
)

velocityGradientCRK = warpOperation(
    currentState,
    OperationProperties(
        kernel = config.kernel,
        operation = WarpOperation.Gradient,
        supportMode = SupportScheme.Scatter, # E.3
        gradientMode = GradientScheme.Difference, # E.3
    ),
    queryValues = currentState.velocities,
    domain = config.domain,
    adjacency = adjacency,
    queryVolumes = apparentVolume,
    crkState= crkState,
)

In [ ]:
trace = torch.einsum('...ii', velocityGradient)

traces = torch.eye(velocityGradient.shape[1], device=velocityGradient.device) * trace.view(-1, 1, 1) / velocityGradient.shape[1]

Shear = (velocityGradient + velocityGradient.transpose(1,2))/2 - traces
Rotation = (velocityGradient - velocityGradient.transpose(1,2)) / 2

In [ ]:
matrix = velocityGradient# - velocityGradientCRK
from warpPlot import visualize, PlottingOptions, PlotScaling, GridVisualization, UniformColorMap, DivergingColorMap, Mapping
markerSize = 2
plotter = visualize(
    particleState = currentState,
    domain = config.domain,
    quantities = {
        "A": matrix[:,0,0],
        "B": matrix[:,0,1],
        "C": matrix[:,1,0],
        "D": matrix[:,1,1],
    },
    plotOptions = {
        "A": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
        ),
        "B": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
        ),
        "C": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
        ),
        "D": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
        ),
    },
    figTitle = "Wave Equation Example",
    mosaic = '''AB
    CD''',
    figsize= (11,10),
    # backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)

# if args.exportImages:
#     plotter.export(f'output/{folderName}/frame_00000.png', dpi = args.figureDpi)

In [ ]:
display(plotter.fig)